# Milestone 3: Google Colab T4 GPU Training for Digital Twin LSTM
## Twin-Guided Explainable Intrusion Detection System (X-IDS)

This notebook trains an LSTM sequence forecasting model on **Normal-only** Edge-IIoTset telemetry, quantizes it to **TFLite (INT8/Dynamic Range)**, and exports `twin_model_quantized.tflite` for edge deployment.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Load sampled dataset
# Upload data/sampled_dataset.csv to Colab environment
df = pd.read_csv('sampled_dataset.csv')
normal_df = df[df['Attack_type'] == 'Normal'].copy().reset_index(drop=True)
feature_cols = [c for c in df.columns if c not in ['Attack_type', 'Attack_label']]

print(f"Normal samples: {len(normal_df)}, Features: {len(feature_cols)}")

# Normalize features
scaler = StandardScaler()
scaled_data = scaler.fit_transform(normal_df[feature_cols].values)

# Build sequence windows (Window size W = 5)
WINDOW_SIZE = 5
X, y = [], []
for i in range(len(scaled_data) - WINDOW_SIZE):
    X.append(scaled_data[i:i + WINDOW_SIZE])
    y.append(scaled_data[i + WINDOW_SIZE])
X, y = np.array(X), np.array(y)
print(f"Sequence input shape: {X.shape}, Target shape: {y.shape}")

In [ ]:
# Build LSTM Twin Model
model = Sequential([
    LSTM(64, activation='tanh', input_shape=(WINDOW_SIZE, len(feature_cols)), return_sequences=False),
    Dropout(0.1),
    Dense(32, activation='relu'),
    Dense(len(feature_cols))
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

# Train model
history = model.fit(
    X, y,
    epochs=20,
    batch_size=64,
    validation_split=0.15,
    verbose=1
)

In [ ]:
# Dynamic Range Quantization to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter.convert()

with open('twin_model_quantized.tflite', 'wb') as f:
    f.write(tflite_quant_model)

print("Exported twin_model_quantized.tflite successfully!")